In [0]:
from pyspark.sql import functions as F


In [0]:
bronze_gtfs_trips = spark.table("bg_traffic.bg_traffic_bronze.gtfs_trips")
bronze_gtfs_trips.display()

In [0]:
required_columns = {
    "route_id","service_id","trip_id","trip_headsign","direction_id","shape_id"
}
missing_columns = required_columns - set(bronze_gtfs_trips.columns)
if missing_columns:
    raise ValueError(
        "GRESKA: Izvorni GTFS trips je promenio strukturu, postoje nedostajuce kolone! "
        )

In [0]:
bronze_gtfs_trips.printSchema()

In [0]:
bronze_gtfs_trips.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in bronze_gtfs_trips.columns]).show()

print(f"Total rows: {bronze_gtfs_trips.count()}")

In [0]:
bronze_gtfs_trips = bronze_gtfs_trips.drop("trip_short_name","block_id","wheelchair_accessible")

### Casting

In [0]:
typed_trips = bronze_gtfs_trips.select(
    F.col("route_id").cast("integer"),
    F.col("service_id").cast("string"),
    F.col("trip_id").cast("string"),
    F.col("trip_headsign").cast("string"),
    F.col("direction_id").cast("integer"),
    F.col("shape_id").cast("string")
)
typed_trips.display()

### Null Handling

In [0]:
typed_trips = typed_trips.fillna({"trip_headsign":"Unknown", "direction_id":0, "shape_id":""})

### Dedup

In [0]:
dedup_trips = typed_trips.dropDuplicates(["trip_id"])

dedup_count = typed_trips.count() - dedup_trips.count()
print(f"Broj duplikata: {dedup_count}")

### Valid

In [0]:
valid_trips = typed_trips.filter(
    (F.col("trip_id").isNotNull()) &
    (F.col("route_id").isNotNull()) 
).withColumn("silver_processed_at", F.current_timestamp())

valid_trips.display()

In [0]:
if valid_trips.isEmpty():
    raise Exception("GRESKA: Silver tabela za upisivanje je prazna nakon ciscenja!")

### Write in silver table

In [0]:
valid_trips.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bg_traffic.bg_traffic_silver.gtfs_trip")